# 🧠 Brain Tumor Classifier v2 — FIXED
### Transfer Learning with EfficientNetB0
**Fix:** Using `image_dataset_from_directory` instead of `ImageDataGenerator` for reliable splits

---
**Classes:** glioma_tumor | meningioma_tumor | normal | pituitary_tumor

## ✅ Step 1 — Set GPU & Install Libraries

In [ ]:
# Make sure GPU is enabled: Runtime → Change runtime type → T4 GPU
!pip install kagglehub -q

import os, json, pickle, shutil, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## ✅ Step 2 — Download Dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download('thomasdubail/brain-tumors-256x256')
print('Downloaded to:', path)
print('Contents:', os.listdir(path))

## ✅ Step 3 — Find Data Directory & Check Classes

In [ ]:
import glob

# Auto-detect directory containing class subfolders
DATA_DIR = None
for d in glob.glob(os.path.join(path, '**'), recursive=True):
    if os.path.isdir(d):
        subdirs = [x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))]
        if len(subdirs) >= 4:
            DATA_DIR = d
            break

if DATA_DIR is None:
    DATA_DIR = path

print('Data directory:', DATA_DIR)

CLASS_NAMES = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])
NUM_CLASSES = len(CLASS_NAMES)

print('\n📊 Class Distribution:')
print('-' * 35)
total = 0
for cls in CLASS_NAMES:
    n = len(os.listdir(os.path.join(DATA_DIR, cls)))
    total += n
    print(f'  {cls:25s}: {n}')
print('-' * 35)
print(f'  Total                    : {total}')
print(f'\nClasses: {CLASS_NAMES}')

## ✅ Step 4 — Manually Split into Train / Val / Test Folders
> This is the KEY FIX — we manually split the data so there's no ambiguity

In [ ]:
SPLIT_DIR  = '/content/brain_tumor_split'
TRAIN_DIR  = os.path.join(SPLIT_DIR, 'train')
VAL_DIR    = os.path.join(SPLIT_DIR, 'val')
TEST_DIR   = os.path.join(SPLIT_DIR, 'test')

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# TEST = remaining 0.15

# Only run split if not already done
if not os.path.exists(SPLIT_DIR):
    print('Splitting dataset...')
    for split in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
        for cls in CLASS_NAMES:
            os.makedirs(os.path.join(split, cls), exist_ok=True)

    for cls in CLASS_NAMES:
        src = os.path.join(DATA_DIR, cls)
        files = [f for f in os.listdir(src) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        random.shuffle(files)

        n       = len(files)
        n_train = int(n * TRAIN_RATIO)
        n_val   = int(n * VAL_RATIO)

        splits = {
            TRAIN_DIR : files[:n_train],
            VAL_DIR   : files[n_train:n_train+n_val],
            TEST_DIR  : files[n_train+n_val:]
        }

        for split_dir, split_files in splits.items():
            for f in split_files:
                shutil.copy(os.path.join(src, f), os.path.join(split_dir, cls, f))

        print(f'  {cls:25s} → train:{n_train}  val:{n_val}  test:{n-n_train-n_val}')

    print('\n✅ Split complete!')
else:
    print('Split already exists, skipping.')

## ✅ Step 5 — Load Datasets with image_dataset_from_directory

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode='categorical'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode='categorical'
)

# Verify class names match expected
CLASS_NAMES = train_ds.class_names
print('Class names:', CLASS_NAMES)
print('Train batches:', len(train_ds))
print('Val   batches:', len(val_ds))
print('Test  batches:', len(test_ds))

## ✅ Step 6 — Augmentation & Performance Tuning

In [ ]:
# Augmentation pipeline (applied only on training data)
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.1),
], name='augmentation')

# Normalize to [0, 1]
normalization = layers.Rescaling(1./255)

def prepare_train(images, labels):
    images = normalization(images)
    images = augment(images, training=True)
    return images, labels

def prepare_eval(images, labels):
    images = normalization(images)
    return images, labels

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(prepare_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(prepare_eval,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(prepare_eval,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

print('✅ Datasets ready')

## ✅ Step 7 — Visualize Sample Images

In [ ]:
# Show one batch of training images
images, labels = next(iter(
    tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, image_size=IMG_SIZE, batch_size=16,
        shuffle=True, seed=SEED, label_mode='categorical'
    )
))

label_indices = tf.argmax(labels, axis=1).numpy()

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].numpy().astype('uint8'))
    ax.set_title(CLASS_NAMES[label_indices[i]], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=120)
plt.show()

## ✅ Step 8 — Build Model (EfficientNetB0 + Custom Head)

In [ ]:
def build_model(num_classes):
    base = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base.trainable = False  # Freeze base initially

    inputs = keras.Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs), base

model, base_model = build_model(NUM_CLASSES)

print(f'Total params    : {model.count_params():,}')
print(f'Trainable params: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')

## ✅ Step 9 — Phase 1 Training (Head Only, 15 Epochs)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cb1 = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_phase1.keras', save_best_only=True, verbose=0)
]

print('🚀 Phase 1 — Training Classification Head')
print('='*50)

h1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=cb1
)

print(f'\nBest Val Accuracy: {max(h1.history["val_accuracy"]):.4f}')

## ✅ Step 10 — Phase 2 Fine-Tuning (Unfreeze Top Layers)

In [ ]:
# Unfreeze top 40 layers of EfficientNetB0
base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False

trainable = sum(1 for l in model.layers if l.trainable)
print(f'Trainable layers: {trainable}')

# Lower learning rate for fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cb2 = [
    EarlyStopping(patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_phase2.keras', save_best_only=True, verbose=0)
]

print('\n🔥 Phase 2 — Fine-Tuning')
print('='*50)

h2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=cb2
)

print(f'\nBest Val Accuracy: {max(h2.history["val_accuracy"]):.4f}')

## ✅ Step 11 — Plot Training Curves

In [ ]:
acc      = h1.history['accuracy']     + h2.history['accuracy']
val_acc  = h1.history['val_accuracy'] + h2.history['val_accuracy']
loss     = h1.history['loss']         + h2.history['loss']
val_loss = h1.history['val_loss']     + h2.history['val_loss']
epochs   = range(1, len(acc)+1)
sep      = len(h1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History', fontsize=15, fontweight='bold')

ax1.plot(epochs, acc,     label='Train Acc',  color='#2ecc71', linewidth=2)
ax1.plot(epochs, val_acc, label='Val Acc',    color='#e74c3c', linewidth=2)
ax1.axvline(sep, color='gray', linestyle='--', alpha=0.7, label='Fine-tune start')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch')
ax1.set_ylim([0, 1.05]); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, loss,     label='Train Loss', color='#3498db', linewidth=2)
ax2.plot(epochs, val_loss, label='Val Loss',   color='#f39c12', linewidth=2)
ax2.axvline(sep, color='gray', linestyle='--', alpha=0.7, label='Fine-tune start')
ax2.set_title('Loss'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print(f'Final Val Accuracy: {val_acc[-1]:.4f}')

## ✅ Step 12 — Evaluate on Test Set

In [ ]:
print('📊 Test Set Evaluation')
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f'\nTest Accuracy : {test_acc:.4f}')
print(f'Test Loss     : {test_loss:.4f}')

In [ ]:
# Confusion matrix & classification report
y_true, y_pred = [], []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds,  axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print('\n📋 Classification Report')
print('='*55)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, linewidths=0.5)
plt.title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## ✅ Step 13 — Save Model & Class Names

In [ ]:
# ---- Save as .h5 (use this in Flask) ----
model.save('brain_tumor_model.h5')
print('✅ Saved: brain_tumor_model.h5')

# ---- Save as .pkl ----
with open('brain_tumor_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('✅ Saved: brain_tumor_model.pkl')

# ---- Save class names ----
with open('class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f)
print('✅ Saved: class_names.json')
print('\nClass names:', CLASS_NAMES)

## ✅ Step 14 — Download All Files

In [ ]:
from google.colab import files

for fname in [
    'brain_tumor_model.h5',
    'brain_tumor_model.pkl',
    'class_names.json',
    'confusion_matrix.png',
    'training_curves.png'
]:
    files.download(fname)
    print(f'📥 Downloading {fname}...')

print('\n✅ All done! Phase 1 complete.')
print('Next → Phase 2: Flask App')

---
## 🎉 Phase 1 Complete!

### ✅ What was fixed vs v1:
| Issue | v1 | v2 (this) |
|---|---|---|
| Data splitting | `validation_split` in ImageDataGenerator (buggy) | Manual 70/15/15 folder split ✅ |
| Augmentation | Inside ImageDataGenerator | Separate Keras layer pipeline ✅ |
| Dataset loading | `flow_from_directory` | `image_dataset_from_directory` ✅ |
| Performance pipeline | Basic | `.prefetch()` + `.map()` with AUTOTUNE ✅ |
| Test evaluation | On val set only | Dedicated test set ✅ |

### Expected accuracy: **85–95%+**